# formal_30_v4 targeted failure diagnosis

This notebook starts a development-only remediation cycle from the frozen `formal_30_v3` evidence. It does not reinterpret v3 as a tunable holdout and does not alter any v3 result.


## Frozen evidence boundary

- Parent snapshot: `data/experiments/snapshots/formal_30_v3_results_20261022_200548`
- Parent files verified by the preparation step: 234
- v3 outcome: 18/30 configured passed, 18/27 completed passed, 9 completed failures, and 3 execution errors.
- v4 is a targeted development batch. Its pass rate must not be reported as an independent generalization result.


In [ ]:
import json
from pathlib import Path

diagnosis_path = Path('data/experiments/analysis/formal_30_v4_planning_20261022/diagnosis.json')
diagnosis = json.loads(diagnosis_path.read_text(encoding='utf-8'))
diagnosis['summary']


## Targeted development samples

| Sample | Complexity | Role | Target defects |
| --- | --- | --- | --- |
| `course_module_accordion` | medium | semantic_reactivity_representative | hidden_template_reactive_dependency, dom_structure_similarity_low |
| `schema_mapping_workbench` | complex | loop_key_and_no_progress_representative | invalid_s_for_key_binding, dom_critical_attribute_mismatch, no_repair_progress |
| `incident_response_board` | complex | state_access_and_no_progress_representative | vue_this_state, dom_critical_attribute_mismatch, no_repair_progress |
| `rule_builder` | complex | prop_default_representative | missing_prop_defaults, dom_structure_similarity_low, dom_critical_attribute_mismatch |
| `habit_week_tracker` | medium | structure_only_representative | dom_structure_similarity_low |
| `playback_queue` | medium | critical_attribute_only_representative | dom_critical_attribute_mismatch |
| `analytics_dashboard` | complex | infrastructure_timeout_representative | generation_timeout |


## Remediation order

1. Reproduce each selected defect from its frozen final artifact and repair history.
2. Add a focused regression test that fails before changing production logic.
3. Fix the narrowest responsible layer: extractor/schema, validator, evaluator, or Prompt.
4. Run existing regression tests and the v4 dry-run.
5. Execute targeted samples incrementally; do not use `--force` unless a deliberate rerun is recorded.
6. Keep the three timeout cases separate from migration-quality failures.
7. After remediation stabilizes, evaluate on another untouched holdout rather than claiming v4 as independent evidence.


In [ ]:
config_path = Path('migration_pipeline/batch_configs/formal_30_v4_targeted.json')
config = json.loads(config_path.read_text(encoding='utf-8'))
[(item['id'], item['selection_role']) for item in config['samples']]


## CourseModuleAccordion first targeted run

Run window: `2026-10-22T21:09:31+08:00` to `2026-10-22T21:11:30+08:00`. The run completed three repair rounds but did not pass. Static validation passed in the final round; visual evaluation retained `dom_structure_similarity_low` with structure similarity `0.8889`, tag sequence similarity `1.0`, text similarity `1.0`, and tree edit distance `2.0`. Total usage was 80,216 tokens.

Compared with v3, the explicit render arguments `openIds`, `completedIds`, and `unfinishedOnly` were restored, eliminating the final `hidden_template_reactive_dependency` errors. The remaining difference was exact child order in the header label: Vue rendered checkbox then text, while San rendered text then checkbox.

A second root cause was found in pipeline wiring. `ValidateStage` already had `dom_child_order_mismatch`, but batch state stored the component name in `source_file` and the actual Vue path in `vue_file_path`; the check only read `source_file` and silently skipped. The validator now consumes `vue_file_path`, and a state-level regression using this exact artifact passes by detecting the mismatch. The first run is frozen at `data/experiments/snapshots/formal_30_v4_course_module_first_run_20261022_211130` before rerun.


In [ ]:
v4_summary_path = Path('data/experiments/batch_reports/formal_30_v4_targeted_summary.json')
v4_summary = json.loads(v4_summary_path.read_text(encoding='utf-8'))
course_result = next(item for item in v4_summary['samples'] if item['id'] == 'course_module_accordion')
{key: course_result[key] for key in [
    'final_passed', 'stop_reason', 'repair_rounds',
    'final_validation_error_codes', 'final_visual_error_codes',
    'final_structure_similarity', 'final_tag_sequence_similarity',
    'final_text_similarity', 'final_tree_edit_distance', 'total_tokens'
]}


## CourseModuleAccordion rerun result

After wiring `vue_file_path` into `ValidateStage`, the deliberate rerun (`2026-10-22T21:20:35+08:00` to `21:21:33+08:00`) detected both `dom_child_order_mismatch` and `missing_prop_defaults` in the initial generation. One repair round passed static and visual evaluation with structure/tag/text similarity all `1.0` and tree edit distance `0.0`. Usage fell from 80,216 tokens in the first run to 38,010 tokens.

The passing evidence is frozen at `data/experiments/snapshots/formal_30_v4_course_module_pass_20261022_212133`. This confirms the validator wiring change made the already-defined DOM-order contract actionable; it remains development-set evidence rather than an independent generalization result.


## SchemaMappingWorkbench diagnosis

The frozen v3 result contained one evaluator false positive and one genuine migration defect. `invalid_s_for_key_binding` was caused by raw regex matching `sourceKey ===` and `targetKey ===` inside class expressions even though every loop already used inline `trackby`. Exact attribute-name parsing now removes that false positive while retaining detection of a real standalone `key=` attribute.

The genuine defect is `dom_critical_attribute_mismatch`: Vue uses `:disabled="previewMode"` on the transform select, but generated San used `disabled="{{ !canConnect || previewMode }}"`, copying the adjacent connect button condition. Generation and repair contracts now explicitly require critical attribute expressions to remain attached to their original element. Re-evaluating the frozen final San after the validator fix yields static pass, while visual evaluation correctly retains only the critical-attribute error (`0.9942` structure similarity, `0.25` tree edit distance).


## SchemaMappingWorkbench targeted result

The v4 run (`2026-10-22T21:30:52+08:00` to `21:32:43+08:00`) passed after one repair round. No `invalid_s_for_key_binding` was reported, and the transform select retained `disabled="{{previewMode}}"` in both initial and repaired San. Final structure/tag/text similarity is `1.0` with tree edit distance `0.0`; total usage is 77,336 tokens. Evidence is frozen at `data/experiments/snapshots/formal_30_v4_schema_mapping_pass_20261022_213243`.


## IncidentResponseBoard diagnosis

The frozen v3 terminal result was an evaluator failure rather than a confirmed migration failure. All four `vue_this_state` matches were legal private timer accesses to `this._timer`. In addition, both lightweight renderers rejected the valid unparenthesized two-alias loop syntax `stage, index in stages`; later expressions such as `item.done` then evaluated outside loop scope and produced a false `checked` mismatch. San compound class interpolation also retained an empty class when every fragment was false.

After fixing private-field recognition, two-alias loops in both renderers, and compound empty-class normalization, the unchanged frozen v3 final San passes static and visual evaluation: 49 Vue nodes and 49 San nodes, all three similarities `1.0`, and tree edit distance `0.0`. The next fresh v4 run validates the corrected end-to-end evaluator; no migration Prompt change is attributed to this sample.


## IncidentResponseBoard first v4 run and lifecycle-alias correction

The first v4 run (`2026-10-22T22:50:53+08:00` to `22:54:32+08:00`) stopped after three repair rounds with `no_repair_progress`. The final visual comparison was already exact: 49 Vue nodes and 49 San nodes, structure/tag/text similarity `1.0`, and tree edit distance `0.0`. The sole final error was `lifecycle_behavior_not_landed`; total usage was 153,688 tokens. The immutable pre-correction evidence is stored at `data/experiments/snapshots/formal_30_v4_incident_first_run_20261022_225432`.

Inspection showed that the generated `attached()` hook preserved the timer and elapsed-state behavior through `const self = this` followed by `self.data.get/set`. The validator only matched direct `this.data.get/set`, so this was a validator false negative rather than a migration failure. A focused regression reproduced the error before the fix. Lifecycle matching now normalizes only explicit local aliases declared as `const`, `let`, or `var` assigned to `this`; the regression and the 36 related tests pass. Re-evaluating the unchanged first-run final San now gives static pass with no validation errors. A deliberate `--force` rerun is still required because the batch checkpoint records the first run as completed.


## IncidentResponseBoard forced rerun result

The deliberate forced rerun (`2026-10-22T23:16:40+08:00` to `23:18:31+08:00`) passed after one repair round. Initial validation reported `invalid_dynamic_attribute_binding`, `dom_child_order_mismatch`, and `missing_prop_defaults`; the initial visual result reported `dom_structure_similarity_low`. The repaired result has no final validation or visual errors, structure/tag/text similarity `1.0`, and tree edit distance `0.0`. Total usage was 74,328 tokens.

Passing evidence is frozen at `data/experiments/snapshots/formal_30_v4_incident_pass_20261022_231831`. Together with the separately frozen failed first run, this preserves the distinction between the original evaluator false negative and the post-correction end-to-end result. This remains development-set remediation evidence, not an independent holdout result.


## RuleBuilder diagnosis and targeted remediation

The frozen v3 run exhausted three repairs. Its final static error was `missing_prop_defaults`; the frozen evaluator reported structure similarity `0.0727`, tag similarity `0.8776`, text similarity `0.8175`, and tree edit distance `51.0`. Replaying the unchanged final San with the current renderers removes the former critical-attribute error but still fails structurally: 55 Vue nodes versus 30 San nodes, structure similarity `0.1455`, tag similarity `0.7059`, text similarity `0.7479`, and tree edit distance `47.0`.

This is a confirmed migration defect. The final San omitted all four prop defaults from `initData()` and used `const { initialGroups, availableFields } = this.data`. San `this.data` is a Data store rather than a plain object, so both variables became undefined; `groups` was then set to an empty array, `selectedGroup` became null, and the group editor and rule subtree disappeared. The previous validator reported the missing defaults but did not identify this direct runtime cause.

The validator now emits `invalid_san_data_destructuring` for direct object destructuring from `this.data`. Generation and repair Prompts, plus the structured repair contract, now require separate `this.data.get(...)` reads and explicitly forbid treating the store as a plain object. Synthetic and frozen-artifact regressions were added; all 38 related tests pass. The next evidence step is a fresh v4 `rule_builder` run.


## RuleBuilder targeted result

The v4 run (`2026-10-22T23:32:02+08:00` to `23:34:52+08:00`) passed after one repair round. The initial generation preserved the four prop defaults and used `this.data.get('initialGroups')`; its five static errors were all `nested_state_mutation`. The repaired result has no final validation or visual errors. Final structure similarity is `0.9909`, tag and text similarity are `1.0`, tree edit distance is `0.5`, and total usage is 107,232 tokens.

The nonzero distance is limited to two semantically equivalent attributes: San explicitly emits `type="text"` on two inputs while Vue omits `type`, whose HTML default is text. There are no missing or extra nodes. Passing evidence is frozen at `data/experiments/snapshots/formal_30_v4_rule_builder_pass_20261022_233452`. This remains development-set remediation evidence rather than an independent holdout result.


## HabitWeekTracker diagnosis

The frozen v3 run exhausted three repairs with only `dom_structure_similarity_low` remaining. Its recorded final structure similarity was `0.0`, tag similarity `0.6818`, text similarity `0.5432`, and tree edit distance `34.5`. The template uses the valid unparenthesized two-alias loop `day, index in dayLabels`, which the old Vue and San lightweight renderers did not expand. This is the same renderer defect identified for IncidentResponseBoard, not a separate migration defect.

Replaying the unchanged frozen final San with the corrected renderers now passes static and visual evaluation exactly: 47 Vue nodes and 47 San nodes, structure/tag/text similarity `1.0`, zero tree edit distance, and no changed, missing, or extra nodes. A frozen-artifact regression was added and all 39 related tests pass. The next evidence step is a fresh v4 `habit_week_tracker` run; no additional Prompt change is attributed to this sample.


## HabitWeekTracker targeted result

The v4 run (`2026-10-22T23:42:12+08:00` to `23:43:18+08:00`) passed after one repair round. Initial generation reported `missing_prop_defaults` and `dom_structure_similarity_low`. The repaired result has no final validation or visual errors, 47 Vue and 47 San nodes, structure/tag/text similarity `1.0`, and zero tree edit distance. Total usage was 41,525 tokens.

Passing evidence is frozen at `data/experiments/snapshots/formal_30_v4_habit_week_tracker_pass_20261022_234318`. This confirms the shared renderer correction while remaining development-set remediation evidence rather than an independent holdout result.


## PlaybackQueue diagnosis

The frozen v3 run exhausted three repairs with only `dom_critical_attribute_mismatch` remaining. Its final San correctly retained `disabled="{{index === 0}}"`, but the old lightweight renderers failed to expand the valid unparenthesized two-alias loop `track, index in queue`. The `index` expression was therefore evaluated outside loop scope and the first move-up button produced a false disabled mismatch.

Replaying the unchanged frozen final San with the corrected renderers now passes static and visual evaluation exactly: 23 Vue nodes and 23 San nodes, structure/tag/text similarity `1.0`, zero tree edit distance, and no changed, missing, or extra nodes. A frozen-artifact regression was added and all 40 related tests pass. The next evidence step is a fresh v4 `playback_queue` run; no additional Prompt change is attributed to this sample.


## PlaybackQueue targeted result

The v4 run (`2026-10-22T23:49:47+08:00` to `23:50:49+08:00`) passed after one repair round. Initial generation already passed visual evaluation exactly and reported only `missing_prop_defaults`; the repaired result has no final validation or visual errors. Final structure, tag, and text similarity are all `1.0`, tree edit distance is `0.0`, and total usage is 38,891 tokens.

Passing evidence is frozen at `data/experiments/snapshots/formal_30_v4_playback_queue_pass_20261022_235049`. This confirms that the former v3 critical-attribute failure was caused by the old renderer's loop-scope handling. The remaining repair concerns prop-default preservation and does not justify attributing a new Prompt change to the renderer issue. This remains development-set remediation evidence rather than an independent holdout result.


## AnalyticsDashboard infrastructure-timeout diagnosis

The v3 run and its permitted unchanged retry both failed during generation at approximately 120 seconds. The same cutoff affected `TaskManager` and `MonitoringDashboard`. `AnalyticsDashboard.vue` is a 1,152-line, 29,515-byte complex component, and no San artifact was produced; therefore this outcome is an infrastructure execution error and provides no evidence about migration quality.

The timeout budgets were inconsistent: `local_server` terminated the upstream LLM read after 120 seconds, while the migration client allowed 300 seconds. For the v4 development remediation only, the inner default is now 240 seconds and the outer default remains 300 seconds, leaving 60 seconds for response and error propagation. `QwenClient` uses the same 240-second default. A regression fixes both values and their ordering; all 36 related `unittest` cases pass. The frozen v3 protocol and its recorded 120-second failures remain unchanged.

The next evidence step is to restart `local_server` so it loads the new timeout, then run only `analytics_dashboard` without `--force`.


## AnalyticsDashboard first completed v4 run

After restarting with the 240-second upstream timeout, the run (`2026-10-23T12:59:17+08:00` to `13:08:53+08:00`) completed generation and three repair rounds instead of timing out. It did not pass: final validation reported `invalid_s_for_key_binding` and `nested_state_mutation`; visual evaluation reported `dom_structure_similarity_low` and `dom_critical_attribute_mismatch`. Recorded structure/tag/text similarity was `0.8421`/`0.9730`/`0.9434`, tree edit distance was `6.0`, and total usage was 291,987 tokens. The pre-correction evidence is frozen at `data/experiments/snapshots/formal_30_v4_analytics_first_run_20261023_130853`.

Diagnosis separated three causes. First, Vue rendering stopped before `mounted`, while San rendering executed `attached`; only San entered the asynchronous loading state before DOM capture. Second, the mutation validator propagated `eventHistory` through `newHistory = [event, ...history]` and incorrectly treated `newHistory.pop()` as mutation of the original store value. Third, every generated version emitted standalone `trackby="stat.id"`; the final repair also restored standalone `key`, rather than using `s-for="stat in statistics trackby stat.id"`. The first two are evaluator/validator defects; the third is a confirmed migration error.

Three focused regressions failed before correction. Vue `mounted` now runs at the same capture phase as San `attached`; spread-containing array literals are recognized as detached copies; and standalone `trackby` is rejected alongside standalone `key`. Generation and repair Prompts now explicitly forbid both standalone forms. All 39 related `unittest` cases pass. Replaying the unchanged frozen final San now yields 38 nodes on both sides, structure/tag/text similarity `0.9934`/`1.0`/`1.0`, and tree edit distance `0.25`; only the confirmed loop binding defect remains. A deliberate `--force` rerun is required after restarting `local_server` because the failed run is already a completed checkpoint.


## AnalyticsDashboard second completed v4 run

After restarting with the inline `trackby` Prompt correction, the second run (`2026-10-23T13:23:58+08:00` to `13:33:24+08:00`) again completed generation and three repair rounds. It did not pass: recorded final validation contained `vue_this_state` plus three `nested_state_mutation` findings, and visual evaluation reported `dom_critical_attribute_mismatch`. Recorded structure/tag/text similarity was `0.9737`/`1.0`/`1.0`, tree edit distance was `1.0`, and total usage was 291,244 tokens. The unchanged pre-correction evidence is frozen at `data/experiments/snapshots/formal_30_v4_analytics_second_run_20261023_133324`; all eight SHA-256 entries were verified.

Diagnosis separated evaluator defects from one migration defect. `this.refreshInterval` is a nonreactive timer handle rather than SSM data state. `newDataCache = {...dataCache}` is a detached copy before local property assignment. The visual evaluator incorrectly treated a custom-component `value` prop as a native form critical attribute, and both lightweight renderers failed to expand self-closing custom-component loops. The confirmed migration defect is binding direction: ordinary Vue `:prop` bindings on `stat-card`, `trend-chart`, `insight-list`, and `activity-table` were emitted as San `{= ... =}` two-way bindings instead of `{{ ... }}` one-way bindings.

Five focused regressions failed before correction. Validation now limits `vue_this_state` to SSM-known reactive fields, recognizes object-spread copies, and reports `invalid_component_prop_two_way_binding`. Critical attributes are scoped to relevant native HTML elements, both renderers expand self-closing loops, and generation/repair Prompts explicitly reserve `{= ... =}` for source Vue `v-model` on native form controls. All 44 related `unittest` cases pass. Replaying the unchanged frozen final San now yields 41 nodes on both sides and passes visual evaluation with structure/tag/text similarity `0.9573`/`1.0`/`1.0` and tree edit distance `1.75`; the only validation error is the confirmed custom-component binding-direction defect. A third deliberate `--force` run is required after restarting `local_server`. This remains development-set remediation evidence, not an independent holdout result.


## AnalyticsDashboard third completed v4 run

The third run (`2026-10-23T13:56:30+08:00` to `14:03:40+08:00`) completed generation and stopped after two identical repair outcomes. Visual equivalence is now exact: structure/tag/text similarity are all `1.0`, tree edit distance is `0.0`, and both sides contain 41 nodes. Static validation retained only `invalid_s_for_key_binding`; total usage was 213,427 tokens. Evidence is frozen at `data/experiments/snapshots/formal_30_v4_analytics_third_run_20261023_140340`, and all seven SHA-256 entries were verified. The stale three-repair artifact from the preceding run is deliberately excluded.

This remaining finding is a confirmed migration error rather than an evaluator defect. The initial San already emitted the correct `s-for="stat in statistics trackby stat.id"` but also retained the redundant standalone `key="{{stat.id}}"`. Both model repair rounds returned the same attribute, producing `no_repair_progress`. Because the Prompt and validator already identify the rule correctly, further Prompt repetition is not the appropriate primary correction.

RepairStage now applies a narrow deterministic normalization only when `invalid_s_for_key_binding` is the sole static error, visual evaluation has no errors, and the standalone `key`/`trackby` expression exactly matches the inline `s-for ... trackby ...` expression. Incomplete loops, non-equivalent expressions, mixed static failures, and visual failures retain the original LLM repair path. A deterministic correction records provider `deterministic` and uses zero model tokens. Four focused regressions failed across the initial implementation and guard-tightening steps; the complete related suite now passes 49 tests. Strict-guard replay of the unchanged frozen final San removes 24 bytes and then passes static and visual evaluation with 41/41 nodes, all similarities `1.0`, and zero tree edit distance. A fourth deliberate `--force` run is required to record the corrected pipeline outcome; no server restart is needed because this fix is migration-side.


## AnalyticsDashboard fourth v4 run and final targeted result

The fourth run (`2026-10-23T15:13:31+08:00` to `15:20:47+08:00`) passed after two repair rounds. Final validation and visual error lists are empty; structure, tag, and text similarity are all `1.0`, tree edit distance is `0.0`, and total usage was 213,848 tokens (62,744 generation and 151,104 repair). The passing evidence is frozen at `data/experiments/snapshots/formal_30_v4_analytics_pass_20261023_152047`, with all seven SHA-256 entries verified.

Both repairs were performed by `qwen-plus`, not by the deterministic normalizer. The initial output had four static errors (`missing_data_types`, `invalid_s_for_key_binding`, `vue_methods_wrapper`, and `event_arguments_not_landed`) plus two San render errors, so the strict deterministic guard correctly remained inactive. The first Qwen repair removed the loop-key error and left only `missing_prop_defaults`; the second Qwen repair resolved that final error. This provenance matters: the passing run must be reported as a two-round LLM repair, while the deterministic branch remains a guarded pipeline capability validated by regression and frozen-artifact replay.

With this result, `formal_30_v4_targeted` is complete at 7/7 passes: all three medium and all four complex development samples passed, with no execution, final validation, or final visual errors. This is development-set remediation evidence derived from v3 failures and must not be presented as independent holdout performance. Generalization requires a separately untouched test set after freezing the v4 implementation.
